# Sci-Fi Extract EDA
Explore the extracted scifi DB (`data/databases/scifi.db`) — novels, films, and authors.

In [ ]:
from __future__ import annotations

import sys
from pathlib import Path

import mwparserfromhell
import numpy as np
import pandas as pd
import seaborn as sns
from IPython.display import display
from matplotlib import pyplot as plt
from sqlalchemy import create_engine

# Make the src package importable from notebooks/
REPO = Path("../").resolve()
if str(REPO / "src") not in sys.path:
    sys.path.insert(0, str(REPO / "src"))

from wiki_dumps.parse.stream import iter_pages_from_xml_bytes

DB_PATH  = REPO / "data/databases/scifi.db"
DUMP_DIR = REPO / "data/dumps"

engine = create_engine(f"sqlite:///{DB_PATH}")
print("DB:", DB_PATH)
print("Dump dir:", DUMP_DIR)

## 1  Load tables

In [ ]:
novels  = pd.read_sql_table("scifi_novels",  engine)
films   = pd.read_sql_table("scifi_films",   engine)
authors = pd.read_sql_table("scifi_authors", engine)

print(f"novels={len(novels):,}  films={len(films):,}  authors={len(authors):,}")

# Check extraction run metadata (written to wiki.db)
try:
    wiki_engine = create_engine(f"sqlite:///{REPO / 'data/databases/wiki.db'}")
    runs = pd.read_sql_table("extraction_runs", wiki_engine)
    display(runs[runs["schema_name"] == "scifi"][["schema_name","pages_processed","pages_matched","started_at","status"]])
except Exception as e:
    print(f"Could not read extraction_runs: {e}")

engine.dispose()

## 2  Novels EDA

In [ ]:
novels.info()
display(novels.head(5))

In [ ]:
# Null rates
(novels.isna().mean() * 100).sort_values(ascending=False).rename("null %").to_frame()

In [ ]:
novels["pub_year"].dropna().astype(int).hist(bins=40, figsize=(12, 4))
plt.title("Novel publication years")
plt.xlabel("Year")
plt.tight_layout()

In [ ]:
# Top genres (pipe-separated)
(
    novels["genres"]
    .dropna()
    .str.split("|")
    .explode()
    .str.strip()
    .str.lower()
    .value_counts()
    .head(20)
    .plot.barh(figsize=(8, 6), title="Top novel genres")
)
plt.tight_layout()

In [ ]:
# Most prolific authors by novel count
(
    novels["author"]
    .dropna()
    .str.split("|")
    .explode()
    .str.strip()
    .value_counts()
    .head(20)
    .plot.barh(figsize=(8, 6), title="Authors with most novel pages")
)
plt.tight_layout()

In [ ]:
# Most common series
novels["series"].dropna().value_counts().head(15)

## 3  Films EDA

In [ ]:
films.info()
display(films.head(5))

In [ ]:
films["release_year"].dropna().astype(int).hist(bins=40, figsize=(12, 4))
plt.title("Film release years")
plt.xlabel("Year")
plt.tight_layout()

In [ ]:
# Most prolific directors
(
    films["director"]
    .dropna()
    .str.split("|")
    .explode()
    .str.strip()
    .value_counts()
    .head(20)
    .plot.barh(figsize=(8, 6), title="Most prolific sci-fi directors")
)
plt.tight_layout()

In [ ]:
# Films by decade
decade = (films["release_year"].dropna().astype(int) // 10 * 10)
decade.value_counts().sort_index().plot.bar(figsize=(12, 4), title="Sci-fi films by decade")
plt.tight_layout()

## 4  Authors EDA

In [ ]:
authors.info()
display(authors.head(5))

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
authors["birth_year"].dropna().astype(int).hist(bins=30, ax=axes[0])
axes[0].set_title("Author birth years")
authors["death_year"].dropna().astype(int).hist(bins=30, ax=axes[1], color="salmon")
axes[1].set_title("Author death years")
plt.tight_layout()

In [ ]:
# Top nationalities
authors["nationality"].dropna().value_counts().head(15).plot.barh(
    figsize=(8, 5), title="Author nationalities (top 15)"
)
plt.tight_layout()

## 5  Cross-table analysis

In [ ]:
# Authors who appear in both the scifi_authors and scifi_novels tables
# (i.e. have a Wikipedia page AND wrote novels we captured)
novel_authors = (
    novels["author"]
    .dropna()
    .str.split("|")
    .explode()
    .str.strip()
    .value_counts()
    .rename("novel_count")
    .reset_index()
    .rename(columns={"index": "author"})
)

merged = novel_authors.merge(
    authors[["title", "birth_year", "death_year", "nationality"]],
    left_on="author",
    right_on="title",
    how="inner",
).drop(columns="title")

print(f"{len(merged):,} authors matched across both tables")
display(merged.sort_values("novel_count", ascending=False).head(20))

In [ ]:
# Novels that were adapted into films (title overlap)
novel_titles = set(novels["title"].str.lower())
adapted = films[films["based_on"].notna()].copy()
print(f"Films with 'based_on' data: {len(adapted):,}")
display(adapted[["title", "director", "release_year", "based_on"]].head(20))

## 6  Wikitext lookup from preprocessed dump

Fetch the raw wikitext for any page by `page_id` — useful for prototyping new fields.

In [ ]:
import bz2

from wiki_dumps.dumps.index import parse_index
from wiki_dumps.dumps.preprocess import load_raw_index, raw_paths

_dump_files = sorted(DUMP_DIR.glob("*-pages-articles-multistream.xml.bz2"))
assert _dump_files, f"No dump found in {DUMP_DIR}"
DUMP_PATH  = _dump_files[-1]
INDEX_PATH = Path(str(DUMP_PATH).replace(
    "-pages-articles-multistream.xml.bz2",
    "-pages-articles-multistream-index.txt.bz2",
))
RAW_PATH, IDX_PATH = raw_paths(DUMP_PATH)

print("Dump: ", DUMP_PATH.name)
print("Raw:  ", RAW_PATH.name, "exists:", RAW_PATH.exists())
print("Index:", IDX_PATH.name, "exists:", IDX_PATH.exists())

In [ ]:
# Load index into sorted numpy arrays for O(log n) page_id lookups
_raw_index = load_raw_index(IDX_PATH)

_pids: list[int] = []
_blks: list[int] = []
block_idx = -1
prev_offset = -1

with bz2.open(INDEX_PATH, "rt", encoding="utf-8") as f:
    for line in f:
        line = line.rstrip("\n")
        fc = line.index(":")
        sc = line.index(":", fc + 1)
        offset  = int(line[:fc])
        page_id = int(line[fc + 1 : sc])
        if offset != prev_offset:
            block_idx += 1
            prev_offset = offset
        _pids.append(page_id)
        _blks.append(block_idx)

_pid_arr = np.array(_pids, dtype=np.int64)
_blk_arr = np.array(_blks, dtype=np.int32)
_sort_order = np.argsort(_pid_arr, kind="stable")
_pid_arr = _pid_arr[_sort_order]
_blk_arr = _blk_arr[_sort_order]
del _pids, _blks, _sort_order

print(f"Index: {len(_pid_arr):,} pages, {_blk_arr.max()+1:,} blocks")
print(f"Memory: {(_pid_arr.nbytes + _blk_arr.nbytes) / 1e6:.1f} MB")

In [ ]:
from wiki_dumps.parse.types import WikiPage


def fetch_page(page_id: int) -> WikiPage | None:
    """Return the WikiPage for *page_id* by seeking into the raw dump."""
    pos = int(np.searchsorted(_pid_arr, page_id))
    if pos >= len(_pid_arr) or _pid_arr[pos] != page_id:
        return None
    block_idx = int(_blk_arr[pos])
    offset, length = _raw_index[block_idx]
    with RAW_PATH.open("rb") as fh:
        fh.seek(offset)
        xml_bytes = fh.read(length)
    for page in iter_pages_from_xml_bytes(xml_bytes):
        if page.page_id == page_id:
            return page
    return None


def fetch_wikitext(page_id: int) -> str:
    page = fetch_page(page_id)
    return page.wikitext if page else ""


# Smoke test on the first novel
sample_pid = int(novels["page_id"].iloc[0])
page = fetch_page(sample_pid)
if page:
    print(f"{page.title!r}  ({len(page.wikitext):,} chars)")
    print(page.wikitext[:500])

## 7  Prototype new extract fields

Grab a sample of novels or films, fetch their wikitext, and try parsing a new field.
Once happy with the logic, copy it into `scifi.py`.

In [ ]:
def _infobox_value(wikicode: object, key: str) -> str | None:
    for tmpl in wikicode.filter_templates():  # type: ignore[attr-defined]
        if tmpl.has(key):
            return str(tmpl.get(key).value).strip() or None
    return None


# ── prototype a new field here ─────────────────────────────────────────────
def extract_new_fields(page_id: int) -> dict:
    wikitext = fetch_wikitext(page_id)
    if not wikitext:
        return {"pages": None, "isbn": None}
    wikicode = mwparserfromhell.parse(wikitext)
    return {
        "pages": _infobox_value(wikicode, "pages"),
        "isbn":  _infobox_value(wikicode, "isbn"),
        # add more keys here to prototype additional fields
    }


# Run on a small sample (one disk seek per page)
SAMPLE_N = 20
sample = novels.head(SAMPLE_N).copy()
new_fields = pd.DataFrame(
    [extract_new_fields(int(pid)) for pid in sample["page_id"]],
    index=sample.index,
)
display(pd.concat([sample[["title", "author", "pub_year"]], new_fields], axis=1))

## 8  Inspect a single page interactively

In [ ]:
# Change TITLE to any novel/film/author title in the DB
TITLE = novels["title"].iloc[0]

row = novels[novels["title"] == TITLE]
if row.empty:
    row = films[films["title"] == TITLE]
if row.empty:
    row = authors[authors["title"] == TITLE]

assert not row.empty, f"{TITLE!r} not found"
page = fetch_page(int(row.iloc[0]["page_id"]))

if page:
    wikicode = mwparserfromhell.parse(page.wikitext)
    print(f"== {page.title} ==")
    print(f"categories: {page.categories[:5]}...")
    print()
    for tmpl in wikicode.filter_templates():
        if "infobox" in str(tmpl.name).lower():
            print(f"Infobox: {tmpl.name.strip()}")
            for param in tmpl.params:
                val = str(param.value).strip()
                if val:
                    print(f"  {param.name.strip()!s:30s} = {val[:80]}")
            break